In [ ]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict
from sksurv.util import Surv
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from Utils.Config import Config, TrialResult
from Utils.utils import (
    set_seed,
    make_surv_y,
    create_features,
    find_ensemble_model,
    get_top_trial_oofs,
    load_config_yaml,
    HORIZONS
)
from Optuna_Experiment import SEEDS, MODEL_TYPES

In [2]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


# Model Ensemble (differnt Config)

In [ ]:
print("SEEDS: ", SEEDS)
print("MODEL TYPES: ", MODEL_TYPES)

In [ ]:


def load_experiment_config(
    seed: int,
    model_type: str,
    trials_root: str = "Trials",
) -> Config:
    model_type = model_type.lower()
    model_dir = os.path.join(trials_root, str(seed), model_type)

    if not os.path.exists(model_dir):
        raise FileNotFoundError(f"Model directory not found: {model_dir}")

    trial_dirs = sorted(
        d for d in os.listdir(model_dir)
        if d.startswith("trial_") and os.path.isdir(os.path.join(model_dir, d))
    )

    if len(trial_dirs) == 0:
        raise FileNotFoundError(f"No trial directories found in: {model_dir}")

    config_path = os.path.join(model_dir, trial_dirs[0], "config.yaml")

    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")

    return load_config_yaml(config_path)

def load_study_from_dir(
    seed: int,
    model_type: str,
    trials_root: str = "Trials",
) -> optuna.Study:
    model_type = model_type.lower()
    journal_path = os.path.join(
        trials_root,
        str(seed),
        f"{model_type}_journal.log",
    )

    storage = JournalStorage(
        JournalFileBackend(journal_path)
    )

    study = optuna.load_study(
        study_name=f"{model_type}_survival_seed_{seed}",
        storage=storage,
    )
    return study

def collect_one_model_top_oofs(
    seed: int,
    model_type: str,
    data: pd.DataFrame,
    horizons: np.ndarray,
    trials_root: str = "Trials",
    out_dir: str = "TOP_OOF",
    top_ratio: float = 0.3,
) -> dict[tuple[int, str, int], dict]:
    
    model_type = model_type.lower()
    set_seed(seed)

    config = load_experiment_config(
        seed=seed,
        model_type=model_type,
        trials_root=trials_root,
    )

    study = load_study_from_dir(
        seed=seed,
        model_type=model_type,
        trials_root=trials_root,
    )

    return get_top_trial_oofs(
        study=study,
        data=data,
        horizons=horizons,
        out_dir=out_dir,
        top_ratio=top_ratio,
        seed=config.seed,
        n_splits=config.cv_n_splits,
        n_repeats=config.cv_n_repeats,
        model_type=config.model_type.lower(),
    )

def collect_top_trial_oofs_from_configs(
    seeds: list[int],
    model_types: list[str],
    train_data: pd.DataFrame,
    horizons: np.ndarray,
    trials_root: str = "Trials",
    out_dir: str = "TOP_OOF",
    top_ratios: dict[str, float] | None = None,
    verbose: bool = True,
) -> dict[tuple[int, str, int], dict]:
    if top_ratios is None:
        top_ratios = {}

    all_oof_result: dict[tuple[int, str, int], dict] = {}

    for seed in seeds:
        for model_type in model_types:
            model_type = model_type.lower()
            top_ratio = top_ratios.get(model_type, 0.3)

            one_result = collect_one_model_top_oofs(
                seed=seed,
                model_type=model_type,
                data=train_data,
                horizons=horizons,
                trials_root=trials_root,
                out_dir=out_dir,
                top_ratio=top_ratio,
            )

            all_oof_result.update(one_result)

            if verbose:
                print(
                    f"[collect] seed={seed}, model={model_type}, "
                    f"added={len(one_result)}, total={len(all_oof_result)}"
                )

    return all_oof_result

In [6]:
full_oof_result = collect_top_trial_oofs_from_configs(
    SEEDS, MODEL_TYPES, train_processed,
    HORIZONS, top_ratios={'gbsa': 0.5, 'coxnet': 0.5}
)
print(list(full_oof_result.keys()))

Saving OOF predictions for top 1 trials to TOP_OOF/42/gbsa...
Best trial value: 0.8635346037582243
Best Tral Number: 0
[collect] seed=42, model=gbsa, added=1, total=1
Saving OOF predictions for top 1 trials to TOP_OOF/42/coxnet...
Best trial value: 0.9610016958192633
Best Tral Number: 0
[collect] seed=42, model=coxnet, added=1, total=2


FileNotFoundError: Model directory not found: Trials/777/gbsa

# Select Ensemble Model

- 대회에 쓰이는 metric을 직접 최적화 하는 방식으로 진행

## 제약조건

1. 전체 trial중 30% (90개)
2. diversity를 어느정도 갖춘 core model [106,152,236,243]으로 시작
3. 이미 선택된 모델들과 corr이 0.995 이상으로 강하다면 skip
4. improve가 너무 적게 상승한다면 noise로 보고 skip
5. max model은 10개 정도 진행

## Config

In [ ]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')
seed = 42
set_seed(seed)

train_df = pd.read_csv(train_path)
train_processed = create_features(train_df)

print("Metadata path: ", metadata_path)
print("Train path: ", train_path)
print("Test path: ", test_path)
print()

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

Metadata path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
Train path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/train.csv
Test path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/test.csv



In [ ]:
# init_model_list = [106,152,236,243]
best_key, best_item = max(
    full_oof_result.items(),
    key=lambda x: x[1]["value"]
)

init_model_list = [best_key]
max_pair_corr = 0.997
max_ensemble_corr = 0.995
min_improvement_score = 0.0001
max_model_num = 10

In [ ]:
# 다른 모델도 후보로 넣으려면 
# .update 이용하여 딕셔너리 추가


ensemble_result = find_ensemble_model(
    oof_result=full_oof_result,
    label=y_full,
    max_ensemble_corr=max_ensemble_corr,
    max_pair_corr=max_pair_corr,
    min_imporvement_score=min_improvement_score,
    max_model_num=max_model_num,
    init_model_list=init_model_list,
    horizons=HORIZONS,
    verbose=True,
)

===== Initial Ensemble =====
Initial models: [(42, 'gbsa', 0)]
Initial hybrid score: 0.865314
Allow duplicate: False
Max select per model: 3
Ensemble Candidate Model: 1

Current ensemble size: 1
Current hybrid score: 0.865314
[Trial (42, 'coxnet', 0)] score=0.932910 improve=+0.067596 pair_corr=0.95737 ens_corr=0.95737 duplicate=False

>>> SELECTED MODEL [2 / 10]
Trial: (42, 'coxnet', 0)
Improvement: +0.067596
New hybrid score: 0.932910

Current ensemble size: 2
Current hybrid score: 0.932910

No candidate model satisfies the conditions.
Max Pair Corr: 0.997
Max Ensemble Corr: 0.995
Min Improve Score: 0.0001
Stopping the search.


===== Final Ensemble =====
Selected models: [(42, 'gbsa', 0), (42, 'coxnet', 0)]
Model counts: {(42, 'gbsa', 0): 1, (42, 'coxnet', 0): 1}
Model weights: {(42, 'gbsa', 0): 0.5, (42, 'coxnet', 0): 0.5}
Final hybrid score: 0.932910
Final C-index: 0.939798
Final mean brier: 0.0700419197656603
